# Amazon Reviews – Problem 2
## Part 1: Exploratory Data Analysis

## 0. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
sns.set_theme(style="whitegrid")

In [ ]:
train_data = pd.read_csv("train_data.csv", low_memory=False)
print("shape of train data:", train_data.shape)
train_data.head()

In [ ]:
title_brand = pd.read_csv("title_brand.csv")
print("shape of title_brand:", title_brand.shape)
title_brand.head()

### Merge `title_brand` with train data
جدول `title_brand` شامل `asin`های تکراری است و مرج مستقیم باعث تکثیر ردیف‌های نظرات می‌شد (تعداد ردیف‌ها از ۸۳۸٬۹۴۴ به ۸۵۵٬۶۷۹ می‌رسید). بنابراین قبل از مرج، `asin`های تکراری را حذف می‌کنیم.

In [ ]:
dup_asins = title_brand["asin"].duplicated().sum()
print("Duplicated asin rows in title_brand:", dup_asins)

# اگر برای یک asin چند برند/عنوان متفاوت ثبت شده باشد، فقط اولین ردیف نگه داشته می‌شود
title_brand_unique = title_brand.drop_duplicates(subset="asin", keep="first")

train_data_merged = train_data.merge(title_brand_unique, on="asin", how="left")

# بررسی صحت مرج: تعداد ردیف‌ها نباید تغییر کند
assert len(train_data_merged) == len(train_data), "Merge changed the number of rows!"
print("Rows after merge:", len(train_data_merged))
train_data_merged.head()

In [ ]:
train_data_merged.info()

# Part 1

## Question 1 – Distribution of `overall`

In [ ]:
plt.figure(figsize=(8, 5))

ax = sns.countplot(x="overall", data=train_data_merged, hue="overall",
                   palette="viridis", legend=False)

plt.title("Distribution of Overall Ratings", fontsize=14, fontweight="bold")
plt.xlabel("Rating (Overall)", fontsize=12)
plt.ylabel("Count (Number of Reviews)", fontsize=12)

for p in ax.patches:
    height = p.get_height()
    ax.annotate(f"{int(height):,}",
                (p.get_x() + p.get_width() / 2., height),
                ha="center", va="bottom", fontsize=10, xytext=(0, 3),
                textcoords="offset points")

plt.tight_layout()
plt.show()

مجموعه داده‌ها متوازن نیست؛ بیشتر داده‌ها متعلق به کلاس پنجم است.

## Question 2 – Word Clouds (Positive / Neutral / Negative)

In [ ]:
def categorize_sentiment(rating):
    if rating in [4, 5]:
        return "Positive"
    elif rating == 3:
        return "Neutral"
    elif rating in [1, 2]:
        return "Negative"
    return np.nan

train_data_merged["sentiment"] = train_data_merged["overall"].apply(categorize_sentiment)

# کلمات زائد: stopword های استاندارد + کلمات پرتکرار و کم‌اطلاعات این دامنه
custom_stopwords = set(STOPWORDS)
custom_stopwords.update([
    "product", "item", "one", "use", "using", "used", "bought",
    "amazon", "get", "will", "really", "much", "even", "time",
    "buy", "ordered", "purchase", "would", "could", "first",
    "review", "thing", "way", "got", "two", "now", "make"
])
# کلمات نفی حفظ می‌شوند چون در تحلیل احساس معنادارند
negation_words = {"not", "no", "never", "nor", "without"}
custom_stopwords = custom_stopwords - negation_words

sample_size = 20000

def build_corpus(sentiment):
    texts = train_data_merged.loc[train_data_merged["sentiment"] == sentiment, "reviewText"].dropna()
    sample = texts.sample(n=min(sample_size, len(texts)), random_state=42)
    return " ".join(sample.astype(str))

def build_cloud(corpus, colormap):
    return WordCloud(width=800, height=500, background_color="white",
                     stopwords=custom_stopwords, colormap=colormap,
                     max_words=100, collocations=True).generate(corpus)

wc_pos = build_cloud(build_corpus("Positive"), "Greens")
wc_neu = build_cloud(build_corpus("Neutral"), "Oranges")
wc_neg = build_cloud(build_corpus("Negative"), "Reds")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, wc, title, color in zip(
    axes,
    [wc_pos, wc_neu, wc_neg],
    ["Positive Reviews (Overall 4 & 5)", "Neutral Reviews (Overall 3)", "Negative Reviews (Overall 1 & 2)"],
    ["green", "darkorange", "darkred"],
):
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(title, fontsize=16, fontweight="bold", color=color)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ۵۰ کلمه برتر هر دسته و اشتراک مثبت و منفی
pos_top_words = set(list(wc_pos.words_.keys())[:50])
neg_top_words = set(list(wc_neg.words_.keys())[:50])

common_words = pos_top_words.intersection(neg_top_words)
print(f"Common words in top-50 of Positive and Negative ({len(common_words)}):")
print(sorted(common_words))

**تفسیر ابر کلمات و کلمات مشترک:**

- **اشتراک موضوعی:** بخش زیادی از کلمات مشترک بین دسته مثبت و منفی نام محصول یا ویژگی آن است (`camera`، `sound`، `cable`، `case`، `screen`، `keyboard`، `laptop`، `speaker`، `phone`، `TV`، …). این یعنی کاربران در هر دو دسته درباره‌ی یک موضوع صحبت می‌کنند و صرفاً دیدن موضوع نظر، جهت احساس را مشخص نمی‌کند.
- **کلمات نفی و احساسی در هر دو دسته:** کلمه‌ی `not` در هر سه ابر یکی از بزرگ‌ترین کلمات است و `no` هم مشترک است. در دسته‌ی مثبت این کلمات معمولاً در عبارت‌هایی مثل «no problem / not bad / does not disappoint» می‌آیند و در دسته‌ی منفی در «does not work / not good». به همین دلیل `problem` و `issue` در دسته‌ی مثبت و `good` و `great` در دسته‌ی منفی هم دیده می‌شوند (مثلاً «not good»). یعنی معنای کلمه به **بافت** وابسته است و حذف کلمات نفی یا استفاده از مدل‌های بدون در نظر گرفتن ترتیب کلمات (bag-of-words ساده) باعث از دست رفتن اطلاعات می‌شود.
- **دسته‌ی مثبت:** کلمات مثبت مشخص مثل `great`، `good`، `love`، `easy`، `nice` و `well` برجسته‌اند.
- **دسته‌ی منفی:** `work` (در قالب «not work»)، `device`، `tried`، `return`، `month`، `day`، `money`، `update` و `never` برجسته‌اند. این‌ها نشان‌دهنده‌ی نارضایتی از خرابی و عملکرد دستگاه، تلاش‌های ناموفق و مرجوع کردن کالاست.
- **دسته‌ی خنثی:** ترکیبی از هر دو طیف است؛ کلمات مثبت (`good`، `great`، `better`، `nice`) و منفی (`issue`، `problem`، `not`، `no`) هم‌زمان دیده می‌شوند و کلماتی مثل `price`، `seem`، `still`، `though` و `fine` هم هست. این با ماهیت نظرات امتیاز ۳ می‌خواند: کاربر هم نکات مثبت و هم نکات منفی را ذکر می‌کند و اغلب در مورد قیمت و عملکرد قابل‌قبول ولی نه عالی نظر می‌دهد.
- **جمع‌بندی:** کلمات مشترک بین مثبت و منفی ثابت می‌کند که برای تحلیل احساس باید کلمات نفی را حذف نکرد و از n-gram یا مدل‌های وابسته به بافت (مثل ترنسفورمرها) استفاده کرد.

## Question 3 – Top 10 Reviewers by Total Votes

In [ ]:
# ستون vote: حذف کاما، تبدیل به عدد و جایگزینی مقادیر خالی با صفر
train_data_merged["vote_clean"] = pd.to_numeric(
    train_data_merged["vote"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
).fillna(0).astype(int)

# نام خالی با شناسه‌ی کاربر جایگزین می‌شود
train_data_merged["reviewerName_clean"] = train_data_merged["reviewerName"].fillna(train_data_merged["reviewerID"])

# گروه‌بندی فقط بر اساس reviewerID تا اگر یک نفر با چند اسم نظر داده، آرایش تقسیم نشود
top_10_voted_reviewers = (
    train_data_merged
    .groupby("reviewerID")
    .agg(
        reviewer_name=("reviewerName_clean", "first"),
        total_votes=("vote_clean", "sum"),
        review_count=("vote_clean", "size"),
    )
    .sort_values("total_votes", ascending=False)
    .head(10)
    .reset_index(drop=True)
    .rename(columns={"reviewer_name": "Reviewer Name",
                     "total_votes": "Total Votes",
                     "review_count": "Total Reviews"})
)
top_10_voted_reviewers

## Question 4 – Review Length Histogram

In [ ]:
train_data_merged["reviewLength"] = train_data_merged["reviewText"].fillna("").str.len()

percentiles = train_data_merged["reviewLength"].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
print(percentiles)

# حدود فیلتر: صدک ۱ تا ۹۹ (حذف دم‌های کم‌تعداد و بسیار طولانی)
low, high = train_data_merged["reviewLength"].quantile([0.01, 0.99])
print(f"\nFilter range (1st-99th percentile): {low:.0f} - {high:.0f} characters")

### Without Filtering:

In [ ]:
plt.figure(figsize=(10, 4.5))
sns.histplot(train_data_merged["reviewLength"], bins=100, color="steelblue")
plt.title("Review Length (raw)")
plt.xlabel("Number of characters")
plt.ylabel("Count")
plt.show()

### With Filtering:

In [ ]:
filtered_length = train_data_merged.loc[
    train_data_merged["reviewLength"].between(low, high), "reviewLength"
]

plt.figure(figsize=(10, 4.5))
sns.histplot(filtered_length, bins=100, color="darkorange")
plt.title(f"Review Length (filtered: {low:.0f} - {high:.0f} characters)")
plt.xlabel("Number of characters")
plt.ylabel("Count")
plt.show()

**آیا باید محدودیتی روی تعداد کاراکترها بگذاریم؟** بله. توزیع طول نظرات به‌شدت راست‌چوله است و تعداد کمی نظر بسیار طولانی وجود دارد.
- نظرات بسیار کوتاه (زیر صدک ۱) اطلاعات کافی برای تشخیص احساس ندارند.
- نظرات بسیار طولانی (بالای صدک ۹۹) هزینه‌ی حافظه و زمان را بالا می‌برند، به‌خصوص برای مدل‌های ترنسفورمری که طول ورودی محدود دارند، و باعث padding زیاد می‌شوند.

بازه‌ی پیشنهادی: **صدک ۱ تا ۹۹** که مقادیر دقیق آن در خروجی بالا چاپ شده است. برای مدل‌های ترنسفورمری می‌توان به‌جای حذف، متن را در حدود ۲۵۶ تا ۵۱۲ توکن truncate کرد.

## Question 5 – Top 10 Products by Number of 5-Star Reviews

In [ ]:
five_star_top10 = (
    train_data_merged[train_data_merged["overall"] == 5]
    .groupby("asin")
    .agg(brand=("brand", "first"),
         title=("title", "first"),
         five_star_count=("asin", "size"))
    .sort_values("five_star_count", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

from html import unescape
five_star_top10["title"] = five_star_top10["title"].apply(lambda t: unescape(t) if isinstance(t, str) else t)

five_star_top10.columns = ["نام برند", "عنوان محصول", "تعداد نظرات پنج‌ستاره"]
five_star_top10.index = range(1, len(five_star_top10) + 1)
five_star_top10.index.name = "رتبه"
five_star_top10

## Question 6 – Top 10 Brands by Review Count & Their Average Rating

In [ ]:
missing_brand = train_data_merged["brand"].isna().sum()
print(f"Reviews without brand (excluded from this analysis): {missing_brand:,} "
      f"({missing_brand / len(train_data_merged):.2%})")

# بررسی املای برندهای پرتکرار برای اطمینان از نبود نام‌های متفاوت برای یک برند
print("\nMost frequent brand names:")
print(train_data_merged["brand"].value_counts().head(20))

In [ ]:
top_brands = train_data_merged["brand"].value_counts().head(10).index

brand_table = (
    train_data_merged[train_data_merged["brand"].isin(top_brands)]
    .groupby("brand")
    .agg(avg_rating=("overall", "mean"), review_count=("overall", "size"))
    .sort_values("avg_rating", ascending=False)
    .reset_index()
)

brand_table["avg_rating"] = brand_table["avg_rating"].round(3)
brand_table.columns = ["نام برند", "میانگین امتیاز", "تعداد نظرات"]
brand_table.index = range(1, len(brand_table) + 1)
brand_table.index.name = "رتبه"
brand_table